In [ ]:
import numpy as np
import pandas as pd
import os

## Configuration

In [ ]:
dataset_name = "SVHN" # DTD | EuroSAT | GTSRB | MNIST | RESISC45 | Stanford_Cars | SUN397 | SVHN
domain = "Base_Fine_Tuned" # Base_Fine_Tuned | Fine_Tuned_Layer_Skipping
model_name = "CLIP_ViT_Vision" # DeiT | CLIP_ViT_Vision | Google_ViT
transformation = ["Standard", "Base_Fine_Tuned_Classifier", "Base_Linear_Probe"] # "Standard" | "Base_Fine_Tuned_Classifier" | "Base_Linear_Probe"
results_path = f"../Data/20%_B_F/Increments/{dataset_name}/{domain}/Entire_Transformation_Matrix_W"
indices = [i for i in range(12)]

## Loading Data

In [ ]:
def sort(name):
    split = name.split("Results_")
    return int(split[1][0])

In [ ]:
# Standard
standard = []

try:
    for filename in os.listdir(f"{results_path}/Standard"):
        if filename in [".DS_Store"]:
            continue
        file_path = os.path.join(f"{results_path}/Standard", filename)
        if os.path.isfile(file_path):
            standard.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{results_path}/Standard' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

print(len(standard))

standard = sorted(standard, key=lambda df: sort(df))
standard = [pd.read_json(i) for i in standard]

In [ ]:
# Fine_Tuned_Classifier
fine_tuned_classifier = []

try:
    for filename in os.listdir(f"{results_path}/Fine_Tuned_Classifier"):
        if filename in [".DS_Store"]:
            continue
        file_path = os.path.join(f"{results_path}/Fine_Tuned_Classifier", filename)
        if os.path.isfile(file_path):
            fine_tuned_classifier.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{results_path}/Standard' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

print(len(fine_tuned_classifier))

fine_tuned_classifier = sorted(fine_tuned_classifier, key=lambda df: sort(df))
fine_tuned_classifier = [pd.read_json(i) for i in fine_tuned_classifier]

In [ ]:
# Linear_Probe
linear_probe = []

try:
    for filename in os.listdir(f"{results_path}/Linear_Probe"):
        if filename in [".DS_Store"]:
            continue
        file_path = os.path.join(f"{results_path}/Linear_Probe", filename)
        if os.path.isfile(file_path):
            linear_probe.append(file_path)
except FileNotFoundError:
    print(f"Error: The Folder '{results_path}/Linear_Probe' was not found.")
except Exception as e:
    print(f"An error occured: {e}")

print(len(linear_probe))

linear_probe = sorted(linear_probe, key=lambda df: sort(df))
linear_probe = [pd.read_json(i) for i in linear_probe]

## Finding Best Numbers

In [ ]:
standard_acc = {i: {j: [] for j in indices} for i in range(5)}
fine_tuned_acc = {i: {j: [] for j in indices} for i in range(5)}
linear_probe_acc = {i: {j: [] for j in indices} for i in range(5)}

In [ ]:
for i in range(5):
    for j in indices:
        standard_acc[i][j].append(standard[i]["Classification_Accuracy"][j])
        fine_tuned_acc[i][j].append(fine_tuned_classifier[i]["Classification_Accuracy"][j])
        linear_probe_acc[i][j].append(linear_probe[i]["Classification_Accuracy"][j])

In [ ]:
def mean_std(set):
    mean = []
    std = []

    for i in indices:
        arr = []
        for j in range(5):
            arr.append(set[j][i])
        arr = np.array(arr)
        mean.append(np.mean(arr))
        std.append(np.std(arr))
    
    return mean, std

In [ ]:
standard_acc_mean, standard_acc_std = mean_std(standard_acc)
fine_tuned_acc_mean, fine_tuned_acc_std = mean_std(fine_tuned_acc)
linear_probe_acc_mean, linear_probe_acc_std = mean_std(linear_probe_acc)

In [ ]:
max_indice = standard_acc_mean.index(max(standard_acc_mean))

In [ ]:
print(standard_acc_std.index(max(standard_acc_std)))

In [ ]:
print(f"Accuracies for {dataset_name}:")
print(f"Best Task Matrix Augmentation Accuracy Layer {max_indice}: {standard_acc_mean[max_indice]} +- {standard_acc_std[max_indice]}")
print(f"Linear Probe Accuracy: {linear_probe_acc_mean[11]} +- {linear_probe_acc_std[11]}")